In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

base_path = '/content/drive/MyDrive/Olist_project'
os.listdir(base_path)

['raw_csv', 'clean_csv', 'notebooks']

In [3]:
raw_path = base_path + '/raw_csv'
os.listdir(raw_path)

['olist_customers.csv',
 'olist_geolocation.csv',
 'olist_order_items.csv',
 'olist_order_payments.csv',
 'olist_order_reviews.csv',
 'olist_orders.csv',
 'olist_products.csv',
 'olist_sellers.csv',
 'product_category_name_translation.csv']

In [4]:
import pandas as pd
orders = pd.read_csv('/content/drive/MyDrive/Olist_project/raw_csv/olist_orders.csv')
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
orders.shape

(99441, 8)

In [6]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [7]:
orders.isna().sum()

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


In [8]:
date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [9]:
orders['approval_delay_hours'] = (orders['order_approved_at'] - orders['order_purchase_timestamp']).dt.total_seconds() / 3600
orders['delivery_time_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['estimated_delivery_gap'] = (orders['order_estimated_delivery_date'] - orders['order_delivered_customer_date']).dt.days
orders['is_delayed'] = orders['estimated_delivery_gap'] < 0

In [10]:
orders[['approval_delay_hours','delivery_time_days','estimated_delivery_gap','is_delayed']].describe()

,approval_delay_hours,delivery_time_days,estimated_delivery_gap
count,99281.000000,96476.000000,96476.000000
mean,10.419094,12.094086,10.876881
std,26.038004,9.551746,10.183854
min,0.000000,0.000000,-189.000000
25%,0.215000,6.000000,6.000000
50%,0.343333,10.000000,11.000000
75%,14.580833,15.000000,16.000000
max,4509.180556,209.000000,146.000000


In [11]:
orders['is_delayed'].value_counts(normalize=True) * 100

,proportion
is_delayed,
False,92.129001
True,7.870999


- Only 7.87% orders are delayed
- 92.13% are on time or early

This tells us three things immediately :
- Olist’s logistics system is largely reliable
- Delivery issues exist, but they are exceptions, not the norm

Those 7.9% delayed orders are high-impact outliers and likely drivers of :
- poor reviews
- refunds
- seller complaints

In [12]:
items = pd.read_csv('/content/drive/MyDrive/Olist_project/raw_csv/olist_order_items.csv')
items.info()
items.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


olist_order_items has no missing values in price or freight, making it reliable for revenue calculations. Most orders contain a single item, but some include multiple items, confirming that revenue exists at the line-item level. Price and freight distributions are right-skewed with extreme outliers, so order-level aggregation is required before analysis.

In [13]:
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], errors='coerce')
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  object        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  object        
 3   seller_id            112650 non-null  object        
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 6.0+ MB


In [14]:
order_agg = (items.groupby('order_id').agg(total_items=('order_item_id', 'count'),order_revenue=('price', 'sum'),total_freight=('freight_value', 'sum')).reset_index())
order_agg.info()
order_agg.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98666 entries, 0 to 98665
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       98666 non-null  object 
 1   total_items    98666 non-null  int64  
 2   order_revenue  98666 non-null  float64
 3   total_freight  98666 non-null  float64
dtypes: float64(2), int64(1), object(1)
memory usage: 3.0+ MB


,total_items,order_revenue,total_freight
count,98666.000000,98666.000000,98666.000000
mean,1.141731,137.754076,22.823562
std,0.538452,210.645145,21.650909
min,1.000000,0.850000,0.000000
25%,1.000000,45.900000,13.850000
50%,1.000000,86.900000,17.170000
75%,1.000000,149.900000,24.040000
max,21.000000,13440.000000,1794.960000


Most orders contain a single item, but a subset includes multiple items, confirming item-level revenue aggregation is required. Order revenue and freight are right-skewed with extreme outliers, so medians and percentiles are more reliable than averages.

In [15]:
orders.to_csv('/content/drive/MyDrive/Olist_project/clean_csv/olist_orders_cleaned.csv',index=False)

In [16]:
clean_path = "/content/drive/MyDrive/Olist_project/clean_csv"
items.to_csv(f"{clean_path}/olist_order_items_cleaned.csv", index=False)
order_agg.to_csv(f"{clean_path}/olist_order_agg_cleaned.csv", index=False)


In [17]:
base_path = "/content/drive/MyDrive/Olist_project/raw_csv"
customers = pd.read_csv(f"{base_path}/olist_customers.csv")
payments = pd.read_csv(f"{base_path}/olist_order_payments.csv")

In [18]:
customers.info()
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null

In [19]:
payments['payment_value'] = payments['payment_value'].astype(float)

In [20]:
customers.to_csv(f"{clean_path}/olist_customers_cleaned.csv", index=False)
payments.to_csv(f"{clean_path}/olist_payments_cleaned.csv", index=False)

In [21]:
customers.info()
customers.isna().sum()
customers.duplicated(subset="customer_id").sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


np.int64(0)

In [22]:
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str)

In [23]:
customers.to_csv('/content/drive/MyDrive/Olist_project/clean_csv/olist_customers_cleaned.csv',index=False)

In [24]:
os.listdir('/content/drive/MyDrive/Olist_project/raw_csv/')

['olist_customers.csv',
 'olist_geolocation.csv',
 'olist_order_items.csv',
 'olist_order_payments.csv',
 'olist_order_reviews.csv',
 'olist_orders.csv',
 'olist_products.csv',
 'olist_sellers.csv',
 'product_category_name_translation.csv']

In [25]:
products = pd.read_csv('/content/drive/MyDrive/Olist_project/raw_csv/olist_products.csv')
products.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [26]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [27]:
products.to_csv('/content/drive/MyDrive/Olist_project/clean_csv/olist_products_cleaned.csv',index=False)


In [28]:
import os
os.listdir('/content/drive/MyDrive/Olist_project/clean_csv')

['olist_orders_cleaned.csv',
 'olist_order_items_cleaned.csv',
 'olist_payments_cleaned.csv',
 'olist_customers_cleaned.csv',
 'olist_order_agg_cleaned.csv',
 'olist_products_cleaned.csv']